In [1]:
import pandas as pd
import numpy as np
import re
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

# **Nivell 1 - EDA estructurat i neteja bàsica**

## **Exercici 1: Exploració inicial del dataset**

In [2]:
df = pd.read_excel("sprint8.xlsx", header=3)
df.shape
df.info()
df.head()
df.duplicated(subset=df.columns.drop("Unnamed: 0")).sum()
df.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  1007 non-null   int64  
 1   Nom                         1006 non-null   object 
 2   Cognoms                     1006 non-null   object 
 3   DNI                         1006 non-null   object 
 4   País d'origen               1006 non-null   object 
 5   Ciutat                      1006 non-null   object 
 6   Dia de Naixement            1006 non-null   float64
 7   Mes de Naixement            1006 non-null   float64
 8   Any de Naixement            1006 non-null   float64
 9   Gènere                      966 non-null    object 
 10  Salari mensual              1006 non-null   object 
 11  Fills                       419 non-null    object 
 12  No Fills                    585 non-null    object 
 13  Grup Professional           1005 

,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
1002,1002,Alba,MuÃ±oz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,1003,Claudia,Gil Ferrer,55297384F,Espanya,ValÃ¨ncia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1006,1006,???,###,XXXX,Narnia,Desconeguda,-99.0,99.0,3000.0,??,molts,NaN,NaN,Z,???,NaN,NaN,NaN,NaN,NaN


- **Quantes files i columnes té el dataset?:** Aquest DataFrame té 1010 files y 20 columnes, previament al "header=3", desprès té 1007 files y 20 columnes.
-  **Quin tipus de dades conté cada columna?**
El dataset conté 20 columnes: 1 de tipus int64, 7 de float64 i 12 d'object.   La columna Unnamed: 0 (int64) és un identificador sense nom.
Les columnes Dia/Mes/Any de Naixement, Nombre_fills, Km_anuals, Consum_mitja_L_100km i Temperatura_mitjana_ciutat són float64. Les columnes Nom, Cognoms, DNI, País d'origen, Ciutat, Gènere, Grup Professional i Càrrec són text (object). Salari mensual és object però hauria de ser numèrica. Fills, No Fills i Te_cotxe són object però haurien de ser booleanes. Les dates de naixement estan dividides en tres columnes separades en lloc d'una sola columna de tipus datetime.
- **Hi ha files duplicades?** Hi ha 4 filas duplicadas (np.int64(4))
- **Les dades semblen coherents a primera vista?** No: el fitxer té una capçalera de títol i files sobrants abans dels noms reals de les columnes, i una columna identificadora sense nom (Unnamed: 0). hi ha caràcters malmesos en noms, ciutats, capçaleres i salaris (MuÃ±oz, ValÃ¨ncia, CÃ rrec, â¬). Registres amb dia -99, mes 99, any 3000, país Narnia, ciutat Desconeguda i textos com ???, ###, XXXX o molts. Gènere barreja valors com Dona, H, D, ?? i Z. Salari mensual és text amb símbol € i separadors, i Fills, No Fills i Te_cotxe són booleanes guardades com a object. La data de naixement està dividida en tres columnes, i Fills i No Fills són redundants. Hi ha files gairebé buides i columnes amb molts nuls, com Km_anuals i Consum_mitja_L_100km.
- **Detectes possibles problemes o inconsistències evidents?:** Hi ha 4 files duplicades que caldrá eliminar. Els nuls de Km_anuals i Consum_mitja_L_100km probablement indiquen "no aplica" (persones sense cotxe), mentre que els de Gènere o Nombre_fills semblen dades perdudes. Barreja d'espais i majúscules (País d'origen) amb guions baixos (Km_anuals).

##**Exercici 2: Validacions bàsiques de les dades i decisions d'eliminació**

S'ha llegit l'Excel indicant header=3, ja que les tres primeres files contenien un títol i files buides abans de la capçalera real amb els noms de columna.

Hi había un problema de codificació en noms, cognoms, ciutats i capçeleres. He creat l funció arreglar_codificacion, que recodifica cada valor de text (encode("latin-1").decode("utf-8")), deixant sense tocar els valors que no són text (nuls) o que no es poden recodificar (gestionat amb try/except). S'ha aplicat a totes les columnes de tipus object i també als noms de les columnes.

Hi había 4 duplicats exactes, passant de 1007 files a 1003 files.


Es comprova la unicitat del DNI: 11 registres tenien DNI duplicat, agrupats en 5 casos. En 3 casos (15909601K, 16618423V, 49529438G) les dades identificatives (nom, any de naixement) coincidien, considerant-se el mateix empleat amb respostes d'enquesta contradictòries; se n'ha conservat només un registre per DNI. En 2 casos (54013571W, 90585882A) les dades identificatives eren completament diferents entre si, indicant que el DNI no és fiable com a identificador; eliminant tots els registres d'aquests DNI en no poder determinar-se a quina persona pertanyen. El dataset passa de 1003 a 990 files.

Per últim; es comprova que el DNI tingui un format vàlid, DNI: 8 dígits + lletra; NIE: lletra + 7 dígits + lletra. Es detecta 2 registres invàlids: una fila completament buida (sense cap dada aprofitable) i una fila amb dades clarament inventades (???, ###, XXXX, país Narnia, any 3000, etc.). Cap dels dos casos és imputable, ja que no falta un valor puntual sinó la pròpia identitat del registre. S'eliminen y es passa de 990 files a 988 files.

**El dataset passa de 1007 a 988 files després de la neteja d'aquest exercici.**

In [3]:
df = pd.read_excel("sprint8.xlsx", header=3)


def arreglar_codificacion(texto):
    if not isinstance(texto, str):
        return texto
    try:
        resultado = texto.encode("latin-1").decode("utf-8")
        return resultado
    except Exception:
        return texto

columnas_texto = df.select_dtypes(include="object").columns

for columna in columnas_texto:
    df[columna] = df[columna].apply(arreglar_codificacion)

df.columns = [arreglar_codificacion(col) for col in df.columns]

df.head(59)


cols = df.columns.drop("Unnamed: 0")
df = df.drop_duplicates(subset=cols)
df.shape

#de 1007 passa a 1003 filas


dnis_personas_distintas = ["54013571W", "90585882A"]
df = df[~df["DNI"].isin(dnis_personas_distintas)]

dnis_misma_persona = ["15909601K", "16618423V", "49529438G"]
df = df.drop_duplicates(subset="DNI", keep="first")

patron_dni = r"^(\d{8}[A-Z]|[A-Z]\d{7}[A-Z])$"
dni_valido = df["DNI"].str.match(patron_dni, na=False)

df = df[dni_valido]

df.loc[(df["No Fills"] == True) & (df["Nombre_fills"].isnull()), "Nombre_fills"] = 0
df = df.drop(columns=["Fills", "No Fills"])

print(df.shape)

df


(988, 18)


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",Grup B,Administratiu,0.0,True,NaN,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,996,Lucía,Roca Vila,21702378D,Espanya,Alacant,32.0,0.0,1940.0,NaN,1158â‚¬,Grup A,Data Analyst,0.0,NaN,NaN,NaN,19.4
997,997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,12.0,10.0,1961.0,Dona,"1,074 â‚¬",Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0
998,998,Chloé,Dubois,66354268T,França,Nice,2.0,12.0,1958.0,H,"1,954 â‚¬",Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.0,12.0,1991.0,H,"2,010 â‚¬",Grup C,Analista,1.0,False,NaN,NaN,-10.0


## Exercici 3: Transformacions necessàries i preparació del dataset

S'ha revisat cada columna categòrica amb value_counts(). Ciutat, País d'origen i Grup Professional ja estaven consistents (una sola forma per valor) i no han requerit canvis.

Gènere: tenia 8 variants per a 4 conceptes reals. S'han unificat amb un diccionari de mapeig (.replace()): D/Dona/F → Dona; H/Home/M → Home; A → Altres; NC (equivalent a "sense dada") → NaN.
Càrrec: tenia 13 variants per a 8 càrrecs reals, per diferències de majúscules o llengua. S'han unificat: Data analyst→Data Analyst, Cap de Projecte/Project lead→Cap de projecte, Analyst→Analista, Admin.→Administratiu.

Unificació de la informació sobre fills

Les columnes Fills, No Fills i Nombre_fills eren redundants (les dues primeres eren booleans complementaris que Nombre_fills ja permetia deduir, amb l'avantatge que aquesta dona el detall numèric).

S'han utilitzat Fills/No Fills per completar Nombre_fills abans d'eliminar-les: als 265 casos amb No Fills=True i Nombre_fills buit, s'ha imputat 0 (se sap amb certesa que no tenen fills).
Als 29 casos restants amb Nombre_fills buit, ni Fills ni No Fills tenien valor, per la qual cosa no hi ha cap indici del nombre de fills; s'han deixat com a nul genuí (no imputable sense criteri).
S'han eliminat les columnes Fills i No Fills, quedant Nombre_fills com a única columna sobre aquest aspecte.

Per cada ciutat, s'han detectat outliers mitjançant el rang interquartílic (IQR): valors per sota de Q1 - 1.5×IQR o per sobre de Q3 + 1.5×IQR es consideren no fiables i es converteixen en nuls (78 valors afectats).
S'ha calculat la mediana de temperatura per ciutat (ja sense outliers), triant la mediana en lloc de la mitjana per ser més robusta davant valors extrems.
S'han imputat tots els nuls (els 67 originals i els 78 outliers detectats) amb la mediana de la seva ciutat corresponent.

La coherència entre Te_cotxe i les dades de cotxe. 4 registres amb Te_cotxe=False tenien consum mitjà registrat sense quilometratge: incoherència, convertida a nul. 4 registres amb Te_cotxe=False tenien Km_anuals=0.0 enlloc de NaN: uniformitzats a NaN. Els casos amb Te_cotxe=True i dades d'ús buides no són incoherències, són nuls genuïns.

Hi había tres casos en text català traduïts a número, i la resta netejada amb regex (només dígits), convertint a nul els valors sense cap dígit. S'ha comprovat que Càrrec i Grup Professional són variables independents, totes dues amb influència real en el salari. S'han imputat els 14 nuls amb la mediana combinada de les variables.

In [11]:
# gènere
mapeo_genere = {
    "D": "Dona",
    "Dona": "Dona",
    "F": "Dona",
    "H": "Home",
    "Home": "Home",
    "M": "Home",
    "A": "Altres",
    "NC": np.nan}

df["Gènere"] = df["Gènere"].replace(mapeo_genere)

# càrrec
mapeo_carrec = {
    "Data analyst": "Data Analyst",
    "Cap de Projecte": "Cap de projecte",
    "Project lead": "Cap de projecte",
    "Analyst": "Analista",
    "Admin.": "Administratiu"}

df["Càrrec"] = df["Càrrec"].replace(mapeo_carrec)

# coherencia cotxe
condicion = (df["Te_cotxe"] == False) & (df["Km_anuals"].isnull()) & (df["Consum_mitja_L_100km"].notna())
df.loc[condicion, "Consum_mitja_L_100km"] = np.nan

condicion = (df["Te_cotxe"] == False) & (df["Km_anuals"] == 0.0)
df.loc[condicion, "Km_anuals"] = np.nan

# temperatura: outliers + imputació per mitjana de ciutat
Q1 = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].transform(lambda x: x.quantile(0.25))
Q3 = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
es_outlier = (df["Temperatura_mitjana_ciutat"] < limite_inferior) | (df["Temperatura_mitjana_ciutat"] > limite_superior)
df.loc[es_outlier, "Temperatura_mitjana_ciutat"] = np.nan
mediana_ciutat = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].transform("median")
df["Temperatura_mitjana_ciutat"] = df["Temperatura_mitjana_ciutat"].fillna(mediana_ciutat)

# Salari: text a número i neteja
mapeo_salarios_texto = {
    "mil vuit-cents": "1800",
    "dos mil": "2000",
    "tres mil": "3000"}
df["Salari mensual"] = df["Salari mensual"].replace(mapeo_salarios_texto)

def limpiar_salario(texto):
    if not isinstance(texto, str):
        return texto
    solo_numeros = re.sub(r"\D", "", texto)
    if solo_numeros == "":
        return np.nan
    return int(solo_numeros)

df["Salari mensual"] = df["Salari mensual"].apply(limpiar_salario)

mediana_carrec_grup = df.groupby(["Càrrec", "Grup Professional"])["Salari mensual"].transform("median")
df["Salari mensual"] = df["Salari mensual"].fillna(mediana_carrec_grup)

display(df)




,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,1469.0,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.10
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,Home,2718.0,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.70
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,Dona,1358.0,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.70
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,Home,1478.0,Grup B,Data Analyst,2.0,False,NaN,NaN,18.30
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,1284.0,Grup B,Administratiu,0.0,True,NaN,11.8,13.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,996,Lucía,Roca Vila,21702378D,Espanya,Alacant,32.0,0.0,1940.0,NaN,1158.0,Grup A,Data Analyst,0.0,NaN,NaN,NaN,19.40
997,997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,12.0,10.0,1961.0,Dona,1074.0,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.00
998,998,Chloé,Dubois,66354268T,França,Nice,2.0,12.0,1958.0,Home,1954.0,Grup B,Data Analyst,1.0,NaN,NaN,NaN,16.75
999,999,Adrià,Vila Díaz,57511543T,Espanya,Palma,31.0,12.0,1991.0,Home,2010.0,Grup C,Analista,1.0,False,NaN,NaN,18.10


# Exercici 4: Primeres anàlisis descriptives

In [5]:
resumen_genere = df.groupby("Gènere")["Salari mensual"].agg(["mean", "median", "min", "max"])
resumen_genere = resumen_genere.sort_values("mean")
display(resumen_genere)

tabla_salario = pd.pivot_table(df, values="Salari mensual", index="Gènere", columns="País d'origen", aggfunc="mean", margins=True)
tabla_salario.style.background_gradient(cmap="Oranges").format(na_rep="-")

,mean,median,min,max
Gènere,,,,
Dona,1479.725599,1341.0,701.0,3564.0
Altres,1690.604167,1205.5,673.0,3600.0
Home,1696.523677,1510.0,726.0,3672.0


País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,All
Gènere,,,,,,
Altres,-,1774.357143,1104.333333,-,-,1690.604167
Dona,1429.909091,1476.608179,1404.829787,1465.475000,1779.750000,1479.725599
Home,2103.818182,1703.675573,1658.645161,1662.351351,1479.000000,1696.523677
All,1598.386364,1575.919940,1490.839506,1540.572165,1650.857143,1569.241361


# Nivell 2 - Anàlisi amb dades enriquides i estructures múltiples

## Exercici 1: Enriquiment mitjançant datasets externs

Primer he carregat el CSV de ciutats i el JSON amb la política salarial. Al CSV he corregit dos noms mal escrits (A Coruña i München) i he unit les dades per ciutat i país, perquè Paris i Valencia apareixen en dos països diferents. Amb el cost de vida he fet tres nivells (low, medium i high), repartint les ciutats en tres grups de la mateixa mida.

Després he calculat l'edat amb l'any 2025. Les edats fora del rang 18-90 anys són errors de dades i les he deixat com a nul. A partir de l'edat he assignat la franja, i amb el nombre de fills he marcat qui en té i qui no.

Amb tot això he buscat al JSON l'increment de cada factor: grup professional, fills, edat i cost de vida. Quan un factor no es pot calcular, per exemple perquè la ciutat no surt al CSV, el valor és 0 i el registre queda marcat amb el motiu. Finalment he sumat els increments, he aplicat el màxim del 15 % i he calculat el salari ajustat. Al final comprovo que no s'ha perdut cap fila.

Els registres que no poden rebre algun ajust són 209 (21 %): 131 no tenen cost de vida perquè la ciutat no surt al CSV, 60 tenen una edat errònia i 29 no tenen el nombre de fills. He decidit aplicar 0 al factor que falta i marcar el registre, perquè així no s'inventa cap increment que no es pugui justificar. Exigir tots els ajustos deixaria 209 registres sense salari ajustat. Imputar la mediana del factor només canviaria l'increment 4,46 €/mes de mitjana. L'efecte és que els registres incomplets tenen un increment mitjà més baix (5,80 % davant 6,61 %), però això ve del 0 aplicat i no d'una diferència real entre persones. En les anàlisis posteriors cal tenir-ho en compte o filtrar amb increment_complet.

In [6]:
# lectura
ciutats = pd.read_csv("ciutats_context.csv", encoding="latin-1")
with open("politica_salarial.json") as f:
    politica = json.load(f)

#Noms i utilitzar ciutat + país com clau
ciutats["Ciutat"] = ciutats["Ciutat"].replace({"A Coru\\u00f1a": "A Coruña", "Munchen": "München"})
ciutats = ciutats.rename(columns={"País": "País d'origen"})
ciutats = ciutats.merge(df[["Ciutat", "País d'origen"]].drop_duplicates())  # solo ciudades del df

# Nivell de cost de vida x tercils
q1, q2 = ciutats["Cost_vida_index"].quantile([1/3, 2/3])
ciutats["Nivell_cost_vida"] = pd.cut(ciutats["Cost_vida_index"], [-np.inf, q1, q2, np.inf],
                                     labels=["low", "medium", "high"])

# Unir amb el dataset principal
df_enr = df.merge(ciutats, on=["Ciutat", "País d'origen"], how="left")

# edat validada i franja edat
df_enr["Edat"] = (2025 - df_enr["Any de Naixement"]).where(lambda e: e.between(18, 90))
df_enr["Franja_edat"] = pd.cut(df_enr["Edat"], [17, 29, 44, 59, 150],
                               labels=["18-29", "30-44", "45-59", "60+"])

# situació familiar
df_enr["Situacio_fills"] = np.where(df_enr["Nombre_fills"].isna(), None,
                                    np.where(df_enr["Nombre_fills"] > 0, "has_children", "no_children"))

# ajust por factor del JSON
df_enr["adj_grup"] = df_enr["Grup Professional"].map(politica["base_increment_by_group"])
df_enr["adj_fills"] = df_enr["Situacio_fills"].map(politica["children_adjustment"])
df_enr["adj_edat"] = df_enr["Franja_edat"].astype(object).map(politica["age_adjustment"])
df_enr["adj_cost"] = df_enr["Nivell_cost_vida"].astype(object).map(politica["cost_of_living_adjustment"])
ajustes = ["adj_grup", "adj_fills", "adj_edat", "adj_cost"]

# registres amd ajust que faltan i motiu
motius = {"adj_grup": "sense grup professional", "adj_fills": "sense Nombre_fills",
          "adj_edat": "edat errònia", "adj_cost": "ciutat sense dades al CSV"}
df_enr["ajustos_absents"] = df_enr[ajustes].isna().sum(axis=1)
df_enr["increment_complet"] = df_enr["ajustos_absents"] == 0
df_enr["motiu_absent"] = ""
for col, motiu in motius.items():
    df_enr.loc[df_enr[col].isna(), "motiu_absent"] += motiu + ", "
df_enr["motiu_absent"] = df_enr["motiu_absent"].str.rstrip(", ")

suma = df_enr[ajustes].sum(axis=1)
df_enr["increment_total_pct"] = suma.clip(upper=politica["rules"]["max_total_increment_percent"])
df_enr["Salari_ajustat"] = df_enr["Salari mensual"] * (1 + df_enr["increment_total_pct"] / 100)

#Comprobacions
print("Filas:", df.shape[0], "->", df_enr.shape[0])
print(df_enr[ajustes].isna().sum())
print(df_enr["ajustos_absents"].value_counts().sort_index())
print(df_enr["motiu_absent"].value_counts())
print("Incremento máximo:", suma.max(), "| Salarios < 800:", (df_enr["Salari mensual"] < politica["rules"]["min_salary_threshold"]).sum())
print("Salario ajustado medio:", round(df_enr["Salari_ajustat"].mean(), 2))

suma_A = df_enr[ajustes].sum(axis=1)
suma_B = df_enr[ajustes].sum(axis=1, skipna=False)
suma_C = df_enr[ajustes].fillna(df_enr[ajustes].median()).sum(axis=1)
inc = df_enr["ajustos_absents"] > 0

print("Registros sin salario ajustado si se exige todo (B):", suma_B.isna().sum())
print("\nIncremento medio (%) según si el registro está completo:")
print(df_enr.groupby("increment_complet")["increment_total_pct"].mean().round(2))

dif = (suma_C[inc] - suma_A[inc]) / 100 * df_enr.loc[inc, "Salari mensual"]
print("\nDiferencia media entre C y A (€/mes):", round(dif.mean(), 2))

Filas: 988 -> 988
adj_grup       0
adj_fills     29
adj_edat      60
adj_cost     131
dtype: int64
ajustos_absents
0    779
1    198
2     11
Name: count, dtype: int64
motiu_absent
                                                 779
ciutat sense dades al CSV                        120
edat errònia                                      52
sense Nombre_fills                                26
edat errònia, ciutat sense dades al CSV            8
sense Nombre_fills, ciutat sense dades al CSV      3
Name: count, dtype: int64
Incremento máximo: 12.5 | Salarios < 800: 30
Salario ajustado medio: 1663.61
Registros sin salario ajustado si se exige todo (B): 209

Incremento medio (%) según si el registro está completo:
increment_complet
False    5.80
True     6.61
Name: increment_total_pct, dtype: float64

Diferencia media entre C y A (€/mes): 4.46


## Exercici 2: Reestructuració del dataset

He passat set valors impossibles de Km_anuals a nul: sis de −50 i un d'1.000.000.
Amb melt(), les vuit columnes numèriques a format llarg, mantenint com a identificadors les dades de la persona (DNI, nom, ciutat, país, grup, gènere, càrrec i franja d'edat). Queden dues columnes noves, Variable i Valor, amb 7.904 files (988 persones per 8 variables). Afegint la categoria i la unitat de cada variable. Clima_tipus queda fora perquè és text.

Hi ha variables similars en columnes diferentes, 3 de salaris, 3 de mobilitat i 2 de context, s'agrupen a la columna Categoria. El format ample dificulta comparar i agregar, perquè cal escriure les columnes una a una i cada una té la seva unitat. En format llarg, un sol groupby calcula les estadístiques de les vuit variables, i la puntuació z permet comparar magnituds diferents.

L'ample és millor per llegir una persona, i el llarg per comparar i fer gràfics. L'inconvenient del llarg és que Valor barreja unitats, així que sempre cal agrupar per Variable. El grup Altres només té 24 persones, i les 62 sense gènere queden fora dels agrupaments.

In [7]:
impossibles = (df_enr["Km_anuals"] <= 0) | (df_enr["Km_anuals"] > 500000)
print("Km corregits:", impossibles.sum())
df_enr.loc[impossibles, "Km_anuals"] = np.nan

print("Km = 80000:", (df_enr["Km_anuals"] == 80000).sum(), "de", df_enr["Km_anuals"].notna().sum())
print("Consum = 0.5:", (df_enr["Consum_mitja_L_100km"] == 0.5).sum())
print("Consum = 25:", (df_enr["Consum_mitja_L_100km"] == 25).sum())
print("Temperatura mínima:", df_enr["Temperatura_mitjana_ciutat"].min())

categories = {"Salari mensual": "Salaris", "Salari_ajustat": "Salaris", "increment_total_pct": "Salaris",
              "Km_anuals": "Mobilitat", "Consum_mitja_L_100km": "Mobilitat", "Transport_public_score": "Mobilitat",
              "Temperatura_mitjana_ciutat": "Context", "Cost_vida_index": "Context"}
unitats = {"Salari mensual": "€/mes", "Salari_ajustat": "€/mes", "increment_total_pct": "%",
           "Km_anuals": "km/any", "Consum_mitja_L_100km": "L/100 km", "Transport_public_score": "puntuació",
           "Temperatura_mitjana_ciutat": "°C", "Cost_vida_index": "índex"}

# Passar a format llarg: columnes_id identifiquen la persona, la resta es mesuren
columnes_id = ["DNI", "Nom", "Ciutat", "País d'origen", "Grup Professional", "Gènere", "Càrrec", "Franja_edat"]
df_llarg = df_enr.melt(id_vars=columnes_id, value_vars=list(categories), var_name="Variable", value_name="Valor")
df_llarg["Categoria"] = df_llarg["Variable"].map(categories)
df_llarg["Unitat"] = df_llarg["Variable"].map(unitats)
print("Forma:", df_llarg.shape)
print("Nuls a Valor:", df_llarg["Valor"].isna().sum())

print(df_llarg.head(10))

# Resum de les 8 variables

print(df_llarg.groupby(["Categoria", "Variable", "Unitat"])["Valor"].agg(["count", "mean", "median", "min", "max"]).round(2))

# Format ample: cal escriure les columnes una a una

print(df_enr.groupby("Gènere")[["Salari mensual", "Km_anuals", "Temperatura_mitjana_ciutat"]].mean().round(2))

# Format llarg: una sola instrucció per a les 8 variables

print(df_llarg.groupby(["Variable", "Gènere"])["Valor"].agg(["count", "mean", "median"]).round(2))

# Comparar magnituds diferents amb la puntuació z
grup = df_llarg.groupby("Variable")["Valor"]
df_llarg["Valor_z"] = (df_llarg["Valor"] - grup.transform("mean")) / grup.transform("std")
print("\nValors z per gènere:")
print(df_llarg.pivot_table(index="Variable", columns="Gènere", values="Valor_z").round(2))

Km corregits: 7
Km = 80000: 73 de 299
Consum = 0.5: 58
Consum = 25: 61
Temperatura mínima: -5.0
Forma: (7904, 12)
Nuls a Valor: 1682
         DNI     Nom                      Ciutat País d'origen  \
0  66722344X   Joana                  Valladolid       Espanya   
1  48840994W    Marc                     Alacant       Espanya   
2  14308421X     Noa                     Alacant       Espanya   
3  58586340F     Pol                     Sevilla       Espanya   
4  82070937P   David                      Bilbao       Espanya   
5  25014631B  Alexia  Las Palmas de Gran Canaria       Espanya   
6  36998038E   David                       Palma       Espanya   
7  47376585K  Carmen                   Barcelona       Espanya   
8  37289414J    Nora                   Stavanger       Noruega   
9  75181648S    Alba  Las Palmas de Gran Canaria       Espanya   

  Grup Professional Gènere                 Càrrec Franja_edat        Variable  \
0            Grup B   Dona        Cap de projecte         6

## Exercici 3: Creació de variables derivades (Feature Engineering)

He creat tres variables noves sobre el dataset ample, combinant columnes que ja tenia.

- Índex de dependència familiar = salari / (fills + 1). Dona el salari que toca a cada membre de la llar. El "+1" evita dividir per zero. És la més útil: baixa de 1.533 € sense fills a 315 € amb quatre. Té dues limitacions: part de la baixada ve de la fórmula, i suposa que aquest salari és l'únic ingrés de la llar.
- Ràtio salari/cost de vida = salari / índex de cost de la ciutat. Aporta poc, perquè té una correlació de 0,97 amb el salari. El cost de vida només va de 60 a 90, i els salaris varien molt més.
- Índex de mobilitat = quilòmetres anuals / temperatura. És la pitjor: té una correlació de 0,73 amb els quilòmetres i de −0,50 amb la temperatura. Les ciutats fredes inflen l'índex encara que la persona condueixi igual, i dividir quilòmetres per graus no té un significat clar.

Un registre de Köln, amb −5 °C, donava un índex negatiu (−8.713). Els altres registres de Köln van de 9,5 a 11,5 °C. Per això, quan la temperatura és 0 o menys, no es calcula l'índex de mobilitat. No he imputat cap valor, perquè no hi ha base per estimar-lo. Els valors alts (fins a 30.769) vénen de ciutats fredes combinades amb molts quilòmetres, i els he deixat. Els nuls venen de les columnes d'origen: 29 de fills, 131 de cost de vida i 690 de quilòmetres o temperatura.

In [8]:
df_enr["Index_dependencia"] = df_enr["Salari mensual"] / (df_enr["Nombre_fills"] + 1)
df_enr["Index_mobilitat"] = df_enr["Km_anuals"] / df_enr["Temperatura_mitjana_ciutat"]
df_enr["Ratio_salari_cost"] = df_enr["Salari mensual"] / df_enr["Cost_vida_index"]

print(df_enr[df_enr["Index_mobilitat"] < 0][["Ciutat", "Km_anuals", "Temperatura_mitjana_ciutat", "Index_mobilitat"]])

# Si la temperatura és 0 o menys, no es calcula l'índex de mobilitat
df_enr.loc[df_enr["Temperatura_mitjana_ciutat"] <= 0, "Index_mobilitat"] = np.nan

print(df_enr[["Index_dependencia", "Index_mobilitat", "Ratio_salari_cost"]].describe().round(2))
print(df_enr[["Index_dependencia", "Index_mobilitat", "Ratio_salari_cost"]].isna().sum())

# útils?
print(df_enr[["Index_mobilitat", "Km_anuals", "Temperatura_mitjana_ciutat"]].corr().round(2))
print(df_enr[["Salari mensual", "Index_dependencia", "Ratio_salari_cost"]].corr().round(2))
print(df_enr.groupby("Nombre_fills")["Index_dependencia"].mean().round(2))
print(df_enr.groupby("Grup Professional")["Ratio_salari_cost"].mean().round(2))

    Ciutat  Km_anuals  Temperatura_mitjana_ciutat  Index_mobilitat
350   Köln    43566.0                        -5.0          -8713.2
       Index_dependencia  Index_mobilitat  Ratio_salari_cost
count             959.00           298.00             857.00
mean             1147.88          3925.52              22.39
std               679.28          3099.48               9.11
min               143.60           165.99               8.76
25%               611.80          1658.32              15.47
50%              1067.67          3578.61              20.15
75%              1525.50          5405.41              26.64
max              3672.00         30769.23              59.23
Index_dependencia     29
Index_mobilitat      690
Ratio_salari_cost    131
dtype: int64
                            Index_mobilitat  Km_anuals  \
Index_mobilitat                        1.00       0.73   
Km_anuals                              0.73       1.00   
Temperatura_mitjana_ciutat            -0.50      -0.04 

## Exercici 4: Exportació i síntesi de resultats

S'exporta les dades per grup professional, perquè és la variable que més separa els salaris i és la que fa servir la política salarial. Amb un bucle es crea quatre fitxers CSV, un per grup. Cada fitxer conté totes les columnes de df_enr. Són les dades detallades. Les files dels quatre fitxers sumen 988, de manera que no s'ha perdut cap registre.

El cinquè fitxer, resum_grups.csv, conté les dades agregades: nombre de treballadors, salari mitjà i edat mediana de cada grup. Separar el detall del resum permet fer servir el resum per a informes i els fitxers detallats per a anàlisis o gràfics.

El salari mitjà puja del grup A (1.109,68 €) al grup D (3.119,51 €). L'edat mediana va de 38 anys al grup D a 47 al grup C. El grup D només té 41 treballadors, així que les seves mitjanes són menys fiables. L'edat mediana no compta les 60 persones amb edat errònia, però el nombre de treballadors sí.

In [9]:
for grup in ["Grup A", "Grup B", "Grup C", "Grup D"]:
    dades = df_enr[df_enr["Grup Professional"] == grup]
    nom = "dades_" + grup.replace(" ", "") + ".csv"
    dades.to_csv(nom, index=False, encoding="utf-8-sig")
    print(nom, dades.shape)

grups = df_enr.groupby("Grup Professional")

resum_grups = pd.DataFrame()
resum_grups["Nombre_treballadors"] = grups.size()
resum_grups["Salari_mitja"] = grups["Salari mensual"].mean().round(2)
resum_grups["Edat_mediana"] = grups["Edat"].median()
resum_grups = resum_grups.reset_index()

# 5e fitxer
resum_grups.to_csv("resum_grups.csv", index=False, encoding="utf-8-sig")

# comprovació
print(resum_grups)
print(pd.read_csv("resum_grups.csv"))

dades_GrupA.csv (457, 37)
dades_GrupB.csv (351, 37)
dades_GrupC.csv (139, 37)
dades_GrupD.csv (41, 37)
  Grup Professional  Nombre_treballadors  Salari_mitja  Edat_mediana
0            Grup A                  457       1109.68          44.0
1            Grup B                  351       1650.85          44.0
2            Grup C                  139       2382.92          47.0
3            Grup D                   41       3119.51          38.0
  Grup Professional  Nombre_treballadors  Salari_mitja  Edat_mediana
0            Grup A                  457       1109.68          44.0
1            Grup B                  351       1650.85          44.0
2            Grup C                  139       2382.92          47.0
3            Grup D                   41       3119.51          38.0
